# 29 — CPU / GPU Scaling

Benchmark ql-jax on CPU vs GPU, measure scaling behaviour, and identify the cross-over point.

| Regime | CPU wins | GPU wins |
|--------|----------|----------|
| BSM scalar | ✓ | |
| BSM 100k batch | | ✓ |
| Heston Fourier | ~ | ~ |
| MC 1M paths | | ✓ |

In [ ]:
import sys, os, time
sys.path.insert(0, os.path.join(os.getcwd(), "..") if not os.getcwd().endswith("ql-jax") else ".")
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

# Detect available backends
cpu_available = True
try:
    jax.devices('gpu')
    gpu_available = True
except RuntimeError:
    gpu_available = False

print(f"CPU: {cpu_available}")
print(f"GPU: {gpu_available}")
if gpu_available:
    print(f"GPU device: {jax.devices('gpu')[0]}")

In [ ]:
from ql_jax.engines.analytic.black_scholes_merton import bsm_price
from ql_jax.engines.analytic.heston import heston_price
from ql_jax.engines.monte_carlo.european_mc import mc_european_bs

---
## 1. Helper: Device-Specific Timing

In [ ]:
def bench(fn, *args, device='cpu', warmup=3, runs=10):
    """Time fn on the given device, returning median ms."""
    dev = jax.devices(device)[0]
    placed = [jax.device_put(a, dev) if isinstance(a, jnp.ndarray) else a for a in args]
    fn_jit = jax.jit(fn, device=dev)
    for _ in range(warmup):
        result = fn_jit(*placed)
        jax.block_until_ready(result)
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        result = fn_jit(*placed)
        jax.block_until_ready(result)
        times.append((time.perf_counter() - t0) * 1000)
    return float(np.median(times))

---
## 2. Scalar Pricing: CPU vs GPU

In [ ]:
def scalar_bsm():
    return bsm_price(100.0, 100.0, 1.0, 0.03, 0.01, 0.2, 1.0)

t_cpu = bench(scalar_bsm, device='cpu')
print(f"Scalar BSM on CPU: {t_cpu:.4f} ms")

if gpu_available:
    t_gpu = bench(scalar_bsm, device='gpu')
    print(f"Scalar BSM on GPU: {t_gpu:.4f} ms")
    print(f"Winner: {'CPU' if t_cpu < t_gpu else 'GPU'} ({min(t_cpu, t_gpu)/max(t_cpu, t_gpu):.1f}× faster)")
else:
    print("(GPU not available — CPU-only benchmarks)")

---
## 3. Batch Scaling: Cross-Over Point

In [ ]:
batch_sizes = [10, 100, 1_000, 10_000, 50_000, 100_000]
cpu_times = []
gpu_times = []

for n in batch_sizes:
    S = jnp.full(n, 100.0)
    K = jnp.linspace(80, 120, n)
    T = jnp.full(n, 1.0)
    r = jnp.full(n, 0.03)
    q = jnp.full(n, 0.01)
    sig = jnp.full(n, 0.2)
    cp = jnp.ones(n)
    
    batch_fn = jax.vmap(bsm_price)
    t = bench(batch_fn, S, K, T, r, q, sig, cp, device='cpu')
    cpu_times.append(t)
    
    if gpu_available:
        t = bench(batch_fn, S, K, T, r, q, sig, cp, device='gpu')
        gpu_times.append(t)

plt.figure(figsize=(8, 5))
plt.plot(batch_sizes, cpu_times, 'o-', label='CPU', color='steelblue')
if gpu_available:
    plt.plot(batch_sizes, gpu_times, 's-', label='GPU', color='coral')
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Batch Size')
plt.ylabel('Time (ms)')
plt.title('BSM Batch Pricing: CPU vs GPU Scaling')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("\nThroughput (prices/ms):")
for i, n in enumerate(batch_sizes):
    msg = f"  N={n:>7,}: CPU {n/cpu_times[i]:>10,.0f}"
    if gpu_available:
        msg += f"  GPU {n/gpu_times[i]:>10,.0f}"
    print(msg)

---
## 4. Monte Carlo Scaling

In [ ]:
mc_paths = [10_000, 50_000, 100_000, 500_000, 1_000_000]
mc_cpu = []
mc_gpu = []

for n_p in mc_paths:
    def mc_fn(n_paths=n_p):
        p, _ = mc_european_bs(100.0, 100.0, 1.0, 0.03, 0.01, 0.2, 1,
                              n_paths=n_paths, n_steps=100)
        return p
    
    t = bench(mc_fn, device='cpu')
    mc_cpu.append(t)
    if gpu_available:
        t = bench(mc_fn, device='gpu')
        mc_gpu.append(t)

plt.figure(figsize=(8, 5))
plt.plot(mc_paths, mc_cpu, 'o-', label='CPU', color='steelblue')
if gpu_available:
    plt.plot(mc_paths, mc_gpu, 's-', label='GPU', color='coral')
plt.xscale('log')
plt.yscale('log')
plt.xlabel('MC Paths')
plt.ylabel('Time (ms)')
plt.title('Monte Carlo Scaling: CPU vs GPU')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## 5. Greeks Throughput

In [ ]:
def greeks_batch(S, K, T, r, q, sigma, cp):
    prices = jax.vmap(bsm_price)(S, K, T, r, q, sigma, cp)
    deltas = jax.vmap(jax.grad(bsm_price, argnums=0))(S, K, T, r, q, sigma, cp)
    vegas = jax.vmap(jax.grad(bsm_price, argnums=5))(S, K, T, r, q, sigma, cp)
    return prices, deltas, vegas

N = 50_000
S = jnp.full(N, 100.0)
K = jnp.linspace(80, 120, N)
T = jnp.full(N, 1.0)
r = jnp.full(N, 0.03)
q = jnp.full(N, 0.01)
sig = jnp.full(N, 0.2)
cp = jnp.ones(N)

t_cpu = bench(greeks_batch, S, K, T, r, q, sig, cp, device='cpu')
print(f"50,000 options × (price + Δ + ν) on CPU: {t_cpu:.1f} ms")
print(f"  Throughput: {N * 3 / t_cpu:,.0f} Greeks/ms")

if gpu_available:
    t_gpu = bench(greeks_batch, S, K, T, r, q, sig, cp, device='gpu')
    print(f"50,000 options × (price + Δ + ν) on GPU: {t_gpu:.1f} ms")
    print(f"  Throughput: {N * 3 / t_gpu:,.0f} Greeks/ms")

---
## 6. Memory Transfer Overhead

In [ ]:
if gpu_available:
    sizes_mb = [0.01, 0.1, 1, 10, 100]
    transfer_times = []
    for mb in sizes_mb:
        n_elements = int(mb * 1024 * 1024 / 8)  # float64
        arr = jnp.ones(n_elements)
        gpu_dev = jax.devices('gpu')[0]
        
        times = []
        for _ in range(20):
            t0 = time.perf_counter()
            arr_gpu = jax.device_put(arr, gpu_dev)
            jax.block_until_ready(arr_gpu)
            times.append((time.perf_counter() - t0) * 1000)
        transfer_times.append(float(np.median(times)))
    
    plt.figure(figsize=(8, 5))
    plt.plot(sizes_mb, transfer_times, 'o-', color='orange')
    plt.xscale('log')
    plt.xlabel('Data Size (MB)')
    plt.ylabel('Transfer Time (ms)')
    plt.title('CPU → GPU Transfer Overhead')
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    print("GPU not available — skipping transfer benchmark")

---
## 7. Exercises

1. **Heston cross-over**: Find the batch size where GPU becomes faster than CPU for Heston pricing.
2. **Mixed precision**: Compare float32 vs float64 throughput on GPU.
3. **Multi-GPU**: If multiple GPUs are available, use `jax.pmap` to distribute batches.

---
*End of Notebook 29*